# Assigment 5
Noah Tingbratt (20050720-3057)

How Have I used AI-tools. AI-tools have been used to information-gain, processing short questions about how to fix some stuffs in the code and in combination with documents get relevant parameters for function and which function that is neassasary. In 5.1 a solution than get test perplexity 107 was implemented with tf without AI-tools but in the convertion to torch AI-tools were used to generate some few classes and especcially to fix so it the all the code run great on a gpu. Most of the code is simple to write on tensorflow but worser gpu inizialations and therefore needed AI-tools in the conversion to torch.

The other part of the subassigment I've used AI-tools more restrictive because there I can use np and understand the code better.

The final solution most importent with the setup of AdamW + SGD wasn't an idea from any sources lika AI. Some of the classes with for exemple warmcosinerestart and locked dropout exist in tensorflow but not in torch and therefor was it generated by in combination with AI.

# 1

In [ ]:
!pip install torch torchvision torchaudio --upgrade
!pip install numpy matplotlib

In [ ]:

import numpy as np
import matplotlib.pyplot as plt
from collections import Counter
import torch
from torch import nn, optim
from torch.amp import GradScaler, autocast
import torch.nn.functional as F

read the ptb files

In [ ]:
def read_file(file_path = "ptb.test.txt"):
  with open(file_path, "r") as file:
    return file.read().replace("\n","<eos>")

In [ ]:
test_set = read_file("ptb.test.txt")
training_set = read_file("ptb.train.txt")
validation_set = read_file("ptb.valid.txt")


split the training set and count words then map the word to correct id ... and map word to id and id to word ...

In [ ]:
tokens_training = training_set.split()


In [ ]:
counter = Counter(tokens_training)
vocabular = sorted(counter.keys())
map_word_to_id = {word: id for id, word in enumerate(vocabular)}
map_id_to_word = {id: word for word, id in map_word_to_id.items()}
vocabular_size = len(vocabular)


In [ ]:
training_set_int = [map_word_to_id[word] for word in tokens_training]
tokens_training = None

tokens_test = test_set.split()
test_set_int = [map_word_to_id[word] for word in tokens_test]
tokens_test = None

tokens_valid = validation_set.split()
valid_set_int = [map_word_to_id[word] for word in tokens_valid]
tokens_valid = None


enable torch params for faster training

In [ ]:
torch.backends.cudnn.benchmark = True

torch.backends.cudnn.deterministic = False

torch.set_default_dtype(torch.float32)

print(f"Using device: {torch.device('cuda' if torch.cuda.is_available() else 'cpu')}")
print(f"cudnn.benchmark enabled: {torch.backends.cudnn.benchmark}")

torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True


Using device: cuda
cudnn.benchmark enabled: True


make list to nupy and more

In [ ]:
train_np = np.array(training_set_int)
val_np = np.array(valid_set_int)
test_np = np.array(test_set_int)

#ckeck for validity
print(train_np.shape)
print(train_np.dtype)

(929589,)
int64


In [ ]:
#%env CUDA_LAUNCH_BLOCKING=1

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

Using device: cuda
GPU: NVIDIA A100-SXM4-40GB


First function mask a dropout for every time step then lstm5 use firs an embeded layer with dropout then to a lstm layer with number of layers more than one layer with dropouts and then a fully connected layer with dropouts. The hidden states are stored and importent with init weights. Bias-gates to one to remember old inputs better and use xaviernorm and ortogonal bases. We tie the weights also to set embedding weight to outputs weight to reduce the number of parameters. Below is a cosine warmup cosine schedular. If step is few we got a linear warmup (only used here) and if step is bigger than ....

In [ ]:
class LockedDropout(nn.Module):
    def __init__(self, dropout=0.5):
        super().__init__()
        self.dropout = dropout

    def forward(self, x):
        # make some masks for dropout
        if not self.training or self.dropout == 0:
            return x
        mask = x.new_empty(x.size(0), 1, x.size(2)).bernoulli_(1 - self.dropout)
        mask = mask / (1 - self.dropout)
        mask = mask.expand_as(x)
        return x * mask

class LSTM5(nn.Module):
    def __init__(self, vocab_size, embed_size=650, hidden_size=650, num_layers=3,
                 embed_dropout=0.3, hidden_dropout=0.4, output_dropout=0.5, tie_weights=True):
        super().__init__()
        # itit everything
        self.encoder = nn.Embedding(vocab_size, embed_size)

        self.embed_dropout = LockedDropout(embed_dropout)
        self.hidden_dropout = LockedDropout(hidden_dropout)
        self.output_dropout = LockedDropout(output_dropout)

        self.hidden_size = hidden_size
        self.num_layers = num_layers

        self.lstm = nn.LSTM(
            input_size=embed_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            dropout=hidden_dropout if num_layers > 1 else 0,
            batch_first=True
        )

        self.decoder = nn.Linear(hidden_size, vocab_size)
        if tie_weights:
            if hidden_size != embed_size:
                raise ValueError("Embed_size != hidden_size")
            self.decoder.weight = self.encoder.weight

        self._init_weights()

    def _init_weights(self):
        # set some good values
        nn.init.uniform_(self.encoder.weight, -0.1, 0.1)
        nn.init.xavier_uniform_(self.decoder.weight)
        if self.decoder.bias is not None:
            nn.init.zeros_(self.decoder.bias)

        for name, param in self.lstm.named_parameters():
            if 'weight_ih' in name:
                nn.init.xavier_uniform_(param)
            elif 'weight_hh' in name:
                nn.init.orthogonal_(param)
            elif 'bias' in name:
                nn.init.zeros_(param)
                n = param.size(0)
                param.data[n//4:n//2].fill_(1.0)

    def forward(self, x, hidden):
        emb = self.encoder(x)
        emb = self.embed_dropout(emb)

        raw_output, hidden = self.lstm(emb, hidden)

        raw_output = self.output_dropout(raw_output)
        output = self.decoder(raw_output.reshape(-1, raw_output.size(2)))

        return output, hidden, raw_output

    def init_hidden(self, batch_size, device):
        h0 = torch.zeros(self.num_layers, batch_size, self.hidden_size, device=device)
        c0 = torch.zeros(self.num_layers, batch_size, self.hidden_size, device=device)
        return (h0, c0)

class WarmupCosineScheduler:
    def __init__(self, optimizer, warmup_steps, total_steps, base_lr, min_lr=1e-5):
        # a warmup cosine function good for AdamW warmup importent
        self.optimizer = optimizer
        self.warmup_steps = warmup_steps
        self.total_steps = total_steps
        self.base_lr = base_lr
        self.min_lr = min_lr
        self.step_num = 0

    def step(self):
        self.step_num += 1

        if self.step_num <= self.warmup_steps:
            lr = self.base_lr * self.step_num / self.warmup_steps
        else:
            progress = (self.step_num - self.warmup_steps) / (self.total_steps - self.warmup_steps)
            lr = self.min_lr + 0.5 * (self.base_lr - self.min_lr) * (1 + np.cos(np.pi * progress))

        for param_group in self.optimizer.param_groups:
            param_group['lr'] = lr
        return lr

the function ar_tar calculate the losses by the mean differences in the raw output - was viewd to help overfitting wen added to the loss-function. Batchify cull make a toal continus batching get batches return the batches from. For evalution we inizilize the hidden state to zero and then we calculate the losses without any gradient updates hidden state and logits, then will the total avargae losses be returned. Then we inizilize the model 650 neurons 3 layer embedded dropout was view to be good to be lower hidden size dropout high was good 0.5 and output dropout was view to be also around 0.5. Importent here very with tied weights then was the model compild and the initial learning was Adamw with a cosine warmup.

In [ ]:
def compute_ar_tar_loss(raw_output, alpha=2e-6, beta=1e-6):
    #compute some more losses for not overfitting
    ar_loss = alpha * raw_output.pow(2).mean()
    tar_loss = 0
    if raw_output.size(1) > 1:
        tar_loss = beta * (raw_output[:, 1:] - raw_output[:, :-1]).pow(2).mean()
    return ar_loss + tar_loss

def batchify_full(data, batch_size):
    # batchify
    nbatch = data.size(0) // batch_size
    data = data[:nbatch * batch_size]
    data = data.view(batch_size, -1).contiguous()
    return data


def get_batches(data, seq_len, stride=0):
    # get some batches with stride for not overfitting
    num_batches = (data.size(1) - 1 - stride) // seq_len
    for i in range(num_batches):
        start = i * seq_len + stride
        x = data[:, start:start + seq_len]
        y = data[:, start + 1:start + 1 + seq_len].reshape(-1)
        yield x, y

def evaluate(model, data_source, loss_function, seq_len=35):
    # Very Important
    model.eval()

    # evaluate the data_source to loss_function
    total_loss = 0.0
    total_tokens = 0
    hidden = model.init_hidden(data_source.size(0), device)

    with torch.no_grad():

        for data, targets in get_batches(data_source, seq_len):
            data, targets = data.to(device), targets.to(device)

            if isinstance(hidden, tuple):
                hidden = (hidden[0].detach(), hidden[1].detach())
            else:
                hidden = [(h.detach(), c.detach()) for (h, c) in hidden]

            output, hidden, _ = model(data, hidden)
            loss = loss_function(output, targets)
            total_loss += loss.item() * targets.numel()
            total_tokens += targets.numel()

    return total_loss / total_tokens

vocab_size = 10000
batch_size = 20
eval_batch_size = 10

#creating torch tensors
train_data = torch.tensor(train_np, dtype=torch.long)
val_data   = torch.tensor(val_np, dtype=torch.long)
test_data  = torch.tensor(test_np, dtype=torch.long)

train_data = batchify_full(train_data, batch_size)
val_data = batchify_full(val_data, eval_batch_size)
test_data = batchify_full(test_data, eval_batch_size)

#create the model
model = LSTM5( vocab_size=vocab_size, embed_size=650, hidden_size=650, num_layers=3, embed_dropout=0.35,  hidden_dropout=0.5, output_dropout=0.5, tie_weights=True).to(device)

# compile it
if hasattr(torch, 'compile'):
    model = torch.compile(model)

print(f"Model has {sum(p.numel() for p in model.parameters()):,} parameters in it")

# the loss function cross entropy
loss_function = nn.CrossEntropyLoss()

# we use AdamW in beginning
optimizer = optim.AdamW(model.parameters(), lr=1e-3, weight_decay=5e-6, betas=(0.9, 0.999))

# number of epochs
epochs = 100
# mean value length
seq_len = 35

steps_per_epoch = (train_data.size(1) - 1) // seq_len
total_steps = steps_per_epoch * epochs

#use a warmup cosine scheduler in beginning
scheduler = WarmupCosineScheduler(optimizer, warmup_steps=int(0.05 * total_steps), total_steps = 5 * total_steps, base_lr=1e-3, min_lr=1e-5 )

scaler = GradScaler()

Model has 16,665,600 parameters in it


Now the fun part. We need to set model to train and then we chanche optimizer at epoch 8 to SGD with momentum and low weight decay. SGD get better results at optimizing after a while but the momentum will drift the solution in beginning. For sgd we use an optimizer with anhelling warm restart with doubeling period important to escape local minima. At epoch 80 we remove the restarter  and we use diffrent sizes of sequence length for geting varying but about 35 +-10 with 10% probability. We use also clipnorm 0.25 for not get to high gradients. Now I have said everything important. And we use holdout validation

In [ ]:
best_val_loss = 100000.
patience_valid = 100
patience_counter = 0

for epoch in range(1, epochs + 1):  # 500 epochs
    #importent for updating now
    model.train()

    # at epoch 8 change to SGD
    if epoch == 8:
      optimizer = optim.SGD(
            model.parameters(),
            lr=2.,
            momentum=0.9,
            weight_decay=5e-6,
            nesterov=True
        )

      # make also a cosine annealing warm restart
      scheduler = optim.lr_scheduler.CosineAnnealingWarmRestarts(
            optimizer, T_0=steps_per_epoch*10, T_mult=2, eta_min=1e-5
      )

    # make a s
    if epoch == 80:
      scheduler = None

    # inizilize the hidden states
    hidden = model.init_hidden(batch_size, device)
    total_loss = 0
    num_batches_for_loss = 0

    # importent to not overfit change the length of input sequence and stride
    current_seq_len = int(seq_len if np.random.random() < 0.75 else np.random.uniform(20, 50))
    stride = np.random.randint(0, current_seq_len -1)

    # loop thrught all moving windows
    for batch_index, (data, targets) in enumerate(get_batches(train_data, current_seq_len, stride)):
        data, targets = data.to(device), targets.to(device)

        if isinstance(hidden, tuple):
            hidden = (hidden[0].detach(), hidden[1].detach())

        optimizer.zero_grad()

        # get the loss
        with autocast(device_type='cuda'):
            output, hidden, raw_output = model(data, hidden)
            loss = loss_function(output, targets)

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)

        # clip grad importent
        torch.nn.utils.clip_grad_norm_(model.parameters(), 0.25)

        #updatate scaler
        scaler.step(optimizer)
        scaler.update()

        #update scgeduler if epoch <70 else we stop using scheduler
        if epoch < 70:
          current_lr = scheduler.step()

        total_loss += loss.item()
        num_batches_for_loss += 1

        if batch_index % 200 == 0:
            print(f'Epoch {epoch:3d}, Batch {batch_index:4d}/{steps_per_epoch}, '+f'Loss: {loss.item():.4f}, LR: '+(f"{current_lr}" if epoch<11 else "same as above"))

    #evaluate
    val_loss = evaluate(model, val_data, loss_function)
    train_loss = total_loss / num_batches_for_loss
    val_ppl = np.exp(val_loss)

    print(f'Epoch {epoch:3d} - Train Loss: {train_loss:.4f}, Val Loss: {val_loss:.4f}, Val PPL: {val_ppl:.2f}')

    # use holdout validation with a maimum of 100 epochs without improvement
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        patience_counter = 0
        torch.save({ 'epoch': epoch, 'model_state_dict': model.state_dict(), 'optimizer_state_dict': optimizer.state_dict(), 'val_loss': val_loss, 'val_ppl': val_ppl,}, 'best_lstm_ptb.pt')
        print(f'New best model saved with (val_loss={val_loss:.4f}, val_ppl={val_ppl:.2f})')
    else:
        patience_counter += 1
        if patience_counter >= patience_valid:
            print(f'Early stopping')
            break

Epoch   1, Batch    0/1327, Loss: 9.2110, LR: 1.5071590052750564e-07
Epoch   1, Batch  200/1327, Loss: 7.1544, LR: 3.029389600602864e-05
Epoch   1, Batch  400/1327, Loss: 6.7157, LR: 6.043707611152977e-05
Epoch   1, Batch  600/1327, Loss: 6.6610, LR: 9.05802562170309e-05
Epoch   1, Batch  800/1327, Loss: 6.5630, LR: 0.00012072343632253204
Epoch   1, Batch 1000/1327, Loss: 6.6302, LR: 0.00015086661642803317
Epoch   1, Batch 1200/1327, Loss: 6.5055, LR: 0.0001810097965335343
Epoch   1 - Train Loss: 6.8903, Val Loss: 6.5789, Val PPL: 719.73
New best model saved with (val_loss=6.5789, val_ppl=719.73)
Epoch   2, Batch    0/1327, Loss: 7.7341, LR: 0.0002001507159005275
Epoch   2, Batch  200/1327, Loss: 6.5459, LR: 0.00023029389600602863
Epoch   2, Batch  400/1327, Loss: 6.6980, LR: 0.00026043707611152976
Epoch   2, Batch  600/1327, Loss: 6.5144, LR: 0.0002905802562170309
Epoch   2, Batch  800/1327, Loss: 6.2489, LR: 0.00032072343632253206
Epoch   2, Batch 1000/1327, Loss: 6.1488, LR: 0.00035

We load the best model from holdout and applay fine tuning on it to get little lower perplexity (fine-tuning with low learningrate and higher weight decay) with erly stopping. This only to get to the lowest point around its minima

In [ ]:
# load the model
model.load_state_dict(torch.load('best_lstm_ptb.pt', weights_only=False)['model_state_dict'])

# get loss
test_loss = evaluate(model, test_data, loss_function)
print(f'Test Perplexity before fine tuning: {torch.exp(torch.tensor(test_loss)):.2f}')

Test Perplexity before fine tuning: 74.74


In [ ]:
# fine tuning for 100 epochs more or if its already the minima (obs from the best validations score already)
extra_fine_tuning = 100

#lower for fine tuning
patience_valid = 15
patience_counter = 0


optimizer = torch.optim.SGD(model.parameters(), lr=1e-3,   momentum=0.9,weight_decay=1e-4)

for epoch in range(101, epochs + 1 + extra_fine_tuning):  # 500 epochs
    #importent for updating now
    model.train()

    # inizilize the hidden states
    hidden = model.init_hidden(batch_size, device)
    total_loss = 0
    num_batches_for_loss = 0

    # importent to not overfit change the length of input sequence and stride
    current_seq_len = int(seq_len if np.random.random() < 0.75 else np.random.uniform(20, 50))
    stride = np.random.randint(0, current_seq_len -1)

    # loop thrught all moving windows
    for batch_index, (data, targets) in enumerate(get_batches(train_data, current_seq_len, stride)):
        data, targets = data.to(device), targets.to(device)

        if isinstance(hidden, tuple):
            hidden = (hidden[0].detach(), hidden[1].detach())

        optimizer.zero_grad()

        # get the loss
        with autocast(device_type='cuda'):
            output, hidden, raw_output = model(data, hidden)
            loss = loss_function(output, targets)

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)

        # clip grad importent
        torch.nn.utils.clip_grad_norm_(model.parameters(), 0.25)

        #updatate scaler
        scaler.step(optimizer)
        scaler.update()

        total_loss += loss.item()
        num_batches_for_loss += 1

    #evaluate
    val_loss = evaluate(model, val_data, loss_function)
    train_loss = total_loss / num_batches_for_loss
    val_ppl = np.exp(val_loss)

    print(f'\tEpoch {epoch:3d} - Train Loss: {train_loss:.4f}, Val Loss: {val_loss:.4f}, Val PPL: {val_ppl:.2f}')

    # use holdout validation with a maimum of 100 epochs without improvement
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        patience_counter = 0
        torch.save({ 'epoch': epoch, 'model_state_dict': model.state_dict(), 'optimizer_state_dict': optimizer.state_dict(), 'val_loss': val_loss, 'val_ppl': val_ppl,}, 'best_lstm_ptb.pt')
        print(f'New best model saved with (val_loss={val_loss:.4f}, val_ppl={val_ppl:.2f})')
    else:
        patience_counter += 1
        if patience_counter >= patience_valid:
            print(f'Early stopping')
            break

	Epoch 101 - Train Loss: 3.7913, Val Loss: 4.3345, Val PPL: 76.29
New best model saved with (val_loss=4.3345, val_ppl=76.29)
	Epoch 102 - Train Loss: 3.7841, Val Loss: 4.3320, Val PPL: 76.10
New best model saved with (val_loss=4.3320, val_ppl=76.10)
	Epoch 103 - Train Loss: 3.7794, Val Loss: 4.3302, Val PPL: 75.96
New best model saved with (val_loss=4.3302, val_ppl=75.96)
	Epoch 104 - Train Loss: 3.7776, Val Loss: 4.3286, Val PPL: 75.83
New best model saved with (val_loss=4.3286, val_ppl=75.83)
	Epoch 105 - Train Loss: 3.7755, Val Loss: 4.3273, Val PPL: 75.74
New best model saved with (val_loss=4.3273, val_ppl=75.74)
	Epoch 106 - Train Loss: 3.7762, Val Loss: 4.3265, Val PPL: 75.68
New best model saved with (val_loss=4.3265, val_ppl=75.68)
	Epoch 107 - Train Loss: 3.7733, Val Loss: 4.3259, Val PPL: 75.64
New best model saved with (val_loss=4.3259, val_ppl=75.64)
	Epoch 108 - Train Loss: 3.7733, Val Loss: 4.3256, Val PPL: 75.61
New best model saved with (val_loss=4.3256, val_ppl=75.61)


In [ ]:
# load the model
model.load_state_dict(torch.load('best_lstm_ptb.pt', weights_only=False)['model_state_dict'])

# get loss
test_loss = evaluate(model, test_data, loss_function)
print(f'Test  after fine-tuning: {torch.exp(torch.tensor(test_loss)):.2f}')

Test  after fine-tuning: 73.50


We se that the best model get a perplexity of 73.5

In [ ]:
if False:
  # save the model
  torch.save(torch.load('best_lstm_ptb.pt', weights_only=False)['model_state_dict'], 'best_lstm_weights_v1.pt')

  #default package i think
  from google.colab import files

  # Download the file
  files.download("best_lstm_weights_v1.pt")

In [ ]:
if False:
  # make a new model
  test_model = LSTM5(    vocab_size=vocab_size,   embed_size=650,    hidden_size=650,    num_layers=3,    embed_dropout=0.35,    hidden_dropout=0.5, output_dropout=0.5,tie_weights=True).to(device)
  if hasattr(torch, 'compile'):
      test_model = torch.compile(test_model)

  # set weights to best values
  test_model.load_state_dict(torch.load('best_lstm_weights_v1.pt', weights_only=True))

  # evaluate test losses
  test_loss = evaluate(test_model, test_data, loss_function)
  print(f'Test Perplexity: {torch.exp(torch.tensor(test_loss)):.2f}')

Lstm need to obtain val_loss below 4.38 for get tes_loss below 4.38. loss 4.38 is equvialent with perplexity of 80. First replaced all the "\n" linespace with "<eos>" then created.

Personally i look a the derivates of the losses by the epoch to understand when overfitting will occour and when the model dosn't learn egnouth and my have higher dropout or would have. Increasing number of tokans help. Most of the work is to not overfit and using reosonable time.

AdamW help with fast convergence with weight decay but will easy overfit. With help of adamw in first 7 epochs with very high dropout and warmup cosine its a clean warmup that stops then. With swithing then to a cosine with restart (importent with restart to escape local minima) and change the optimizer to the greater explorer SGD (stochastic gradient decent). The SGD will not overfit the data to fast and with help of also a modified loss function that will add losses to high weights will it work good. The periods of the cosine restart are (10 and 20 and 40 epoch). After 70 epochs will the cosine-restart stop to not escape again and have a steady low learningrate with holdout validation.

This is good to fast convergence with AdamW because SGD can't navigate good with derivates and with the high dropout. AdamW can navigate good and isn't as affected by the learningrate as long as it's low and weights decay isn't to big. We give a great initial weight to SGD with AdamW can we say and it's importent to stop fast for SGD so that the validationloss is about the same as the training-loos so AdamW hav't start to overfit the data. Therefore was epoch 8 set. The validation gap should be small. This setup will on a A100 (I paid google colab much) reach validation perplexity < 80 under 15 minutes and still improving.

The initial weights is set to good weights.

More layers and more units increase the capacity but the dropout need to be increased.

The lstm use diffrent input length and strides (initial words will not be overrepresented) to mitage the overfitting.

Most of this implementation was done in tensorflow but there without adamw but purly sgd we obtain a minimum of 108 perplexity but then in the swith to torch was some ai-tools used to convert to torch and to look similar to that.

Locked dropouts were also used.

After 100 epoch a lower learningrate on the best model with validation score were used (observe) that we test also the model before holdout validation. This is for going back and use a lower learningrate to not overfit if possible around a local optima in validation (this lowered the perplexity about ~1 with SGD on validation score and litle more than 1 on test perplexity). The low learning and weight decay is good to generalize the model to move to the local optima near the optima reached before not to reach a new optima but be near the optima alrady reached.

### I reached 73.5 test perplexity in about 30 minutes (22 minites first, 8 minutes tuning to little better)

In [ ]:
# 4

# 2

Here we use ngrams to calculate perplexity first generate al ngrams in first function and in second one we generate the probability from counts with alpha = 0.0001 add one count for being false to each probability. This lowered the preplexity little and made the calculation numerical stable. Then we used a function to create the perplexity and then some stright forward functions. Function minimize perplexity minimize the perplexity given three vectors and generates best c values given the probabilities and minimized perplexity. get_ngrams_prob_vector only return the unipart, bia-part and tri-part from the index at a position in the words.

In [ ]:

def get_ngrams(words):
    length_word = len(words)

    unigrams = Counter()
    for i in range(length_word):
        unigrams[(words[i])] += 1

    bigrams = Counter()
    for i in range(length_word - 1):
        bigrams[(words[i], words[i+1])] += 1

    trigrams = Counter()
    for i in range(length_word  - 2):
        trigrams[(words[i], words[i+1], words[i+2])] += 1

    return unigrams, bigrams, trigrams

def get_ngrams_prob_vector(words, unigrams, bigrams, trigrams, voc_size=10000):
  vector_tri = np.zeros(len(words) -2, dtype=np.float64)
  vector_bia = np.zeros(len(words) -2, dtype=np.float64)
  vector_uni = np.zeros(len(words) -2, dtype=np.float64)
  length_words = len(words)
  sum_unigram =   np.sum([j for i, j in unigrams.items()])

  for index in range(len(words) - 2):
    word_1, word_2, word_3 = words[index], words[index + 1], words[index + 2]

    alpha = 0.0001
    vector_uni[index] = (unigrams[(word_3)] + alpha)/(sum_unigram + alpha * voc_size )

    alpha = 0.0001
    vector_bia[index] = (bigrams.get((word_2, word_3), 0) + alpha) / (unigrams.get((word_2), 0) +alpha * voc_size)

    alpha = 0.0001
    vector_tri[index] = (trigrams.get((word_1, word_2, word_3), 0) + alpha) / (bigrams.get((word_1, word_2), 0) + alpha * voc_size)

    if vector_uni[index] >1 or vector_bia[index] > 1 or vector_tri[index] > 1:
      print("error:", vector_uni[index], vector_bia[index], vector_tri[index], bigrams.get((word_2, word_3)), bigrams.get(word_2))
  return vector_uni, vector_bia, vector_tri

def get_perplexity(c_1, c_2, vector_uni, vector_bia, vector_tri):
        perplexity = np.exp(- np.sum(np.log(c_1 * vector_tri + c_2 * vector_bia + (1 - c_1 - c_2) * vector_uni))/vector_tri.shape[0])
        return perplexity

def minimize_perplexity(vector_uni, vector_bia, vector_tri):
  N = vector_uni.shape[0]
  best_perplexity = 1e8
  best_cs = [-1, -1]
  for c_i in np.arange(0, 1.0005, 0.0025):
    for c_2 in np.arange(0, 1.0005-c_i, 0.0025):
      c_1 = 1 - c_i - c_2
      summa = c_1 * vector_tri + c_2 * vector_bia + (1 - c_1 - c_2) * vector_uni
      perplexity = np.exp(- np.sum(np.log(summa))/N)

      #print(f"c_1 = {c_1:.2f}, c_2 = {c_2:.2f} perplexity = {perplexity:.4f}")

      if perplexity < best_perplexity:
        best_cs = [c_1, c_2]
        best_perplexity = perplexity
  return best_perplexity, best_cs


create the grams and get perplexity and best c. Thought about a bin search for bining approporate c and c2

In [ ]:

unigrams_v, bigrams_v, trigrams_v = get_ngrams(training_set_int)

vector_uni_valid, vector_bia_valid, vector_tri_valid = get_ngrams_prob_vector(valid_set_int, unigrams_v, bigrams_v, trigrams_v)

best_perplexity, best_cs = minimize_perplexity(vector_uni_valid, vector_bia_valid, vector_tri_valid)

best_c1, best_c2 = best_cs[0], best_cs[1]

vector_uni_t, vector_bia_t, vector_tri_t= get_ngrams_prob_vector(test_set_int, unigrams_v, bigrams_v, trigrams_v)

print("best vali", best_c1, best_c2)
print("perplexity: ", get_perplexity(best_c1, best_c2, vector_uni_t, vector_bia_t, vector_tri_t))

best vali 0.265 0.495
perplexity:  176.86544261035888


Got ~176.86 with alpha = 0.0001. Improves else we set probability to 0 for words never seen.

# 3

In get probability on third part i looked at the distributation of bingrams and implemented it accordingly first i took wrong on the that you should look at the previus biagram without the same word in it. get prob tri generates given words and c values the probabilities acording to part 3. Below is all code for the part 3 which use bins for different count of the firsts biagrams to use different c:s.

In [ ]:
def get_prob_tri(words, c_values, unigrams, bigrams, trigrams, voc_size=10000):
  bins = 13
  vector_prob_tri = np.zeros(len(words) -2, dtype=np.float64)

  length_words = len(words)
  sum_unigram =   np.sum([j for i, j in unigrams.items()])

  for index in range(len(words) - 2):
    word_1, word_2, word_3 = words[index], words[index + 1], words[index + 2]
    alpha = 0.0001
    uni = (unigrams[(word_3)] + alpha)/(sum_unigram + alpha * voc_size )


    bia = (bigrams.get((word_2, word_3), 0) + alpha) / (unigrams.get((word_2), 0) +alpha * voc_size)

    tria = (trigrams.get((word_1, word_2, word_3), 0) + alpha) / (bigrams.get((word_1, word_2), 0) + alpha * voc_size)

    bigram_count = bigrams.get((word_1, word_2), 0)
    if bigram_count < 1:
      bin = 0
    elif bigram_count < 3:
      bin = 1
    elif bigram_count < 5:
      bin = 2
    elif bigram_count < 16:
      bin = 3
    elif bigram_count < 32:
      bin = 4
    elif bigram_count < 64:
      bin = 5
    elif bigram_count < 128:
      bin = 6
    elif bigram_count < 256:
      bin = 7
    elif bigram_count < 512:
      bin = 8
    elif bigram_count < 1500:
      bin = 9
    elif bigram_count < 4000:
      bin = 10
    elif bigram_count < 7150:
      bin = 11
    elif bigram_count >= 7150:
      bin = 12
    else:
      print("error",bigram_count)
    vector_prob_tri[index] = c_values[bin, 0] * tria + c_values[bin, 1] * bia + (1 - c_values[bin, 1] - c_values[bin, 0]) * uni

    if uni >1 or bia > 1 or tria > 1 or vector_prob_tri[index]>1:
      print("error:",vector_prob_tri[index])
  return vector_prob_tri

def get_perplexity_from_probs(probs):
        perplexity = np.exp(- np.sum(np.log(probs))/probs.shape[0])
        return perplexity

get_ngrams_prob_vector_with_map generates three list of list of a type of ngram in a bin bu binning . minimize each c in list with a gready method. Most functions here can be seen as a wrapper to functions above.

My thoughts is explained in code here. get_ngrams_prob_vector_with_map bin each of the indexes with unigram part, biagram part and tria gram part into diffrent arrays in a list by counts of (word_-2, word_-1) from word_0.

Minimize c takes the binned arrays and extract one at a time and call an optimizer for c as in part 2 and then makes a matrix of c_1. values.

In [ ]:
def get_ngrams_prob_vector_with_map(words, unigrams, bigrams, trigrams, voc_size=10000):
  bins = 13
  vector_tri = [[ ] for i in range(bins)]
  vector_bia =  [[ ] for i in range(bins)]
  vector_uni =  [[ ] for i in range(bins)]
  length_words = len(words)
  sum_unigram =   np.sum([j for i, j in unigrams.items()])

  for index in range(len(words) - 2):
    word_1, word_2, word_3 = words[index], words[index + 1], words[index + 2]
    alpha = 0.0001 # one probability for smothing and it means that the count for all other words is one ("imporves valid perplexity very little")
    uni = (unigrams[(word_3)] + alpha)/(sum_unigram + alpha * voc_size )

    bia = (bigrams.get((word_2, word_3), 0) + alpha) / (unigrams.get((word_2), 0) +alpha * voc_size)

    tria = (trigrams.get((word_1, word_2, word_3), 0) + alpha) / (bigrams.get((word_1, word_2), 0) + alpha * voc_size)

    bigram_count = bigrams.get((word_1, word_2), 0)

    if bigram_count < 1:
      bin = 0
    elif bigram_count < 3:
      bin = 1
    elif bigram_count < 5:
      bin = 2
    elif bigram_count < 16:
      bin = 3
    elif bigram_count < 32:
      bin = 4
    elif bigram_count < 64:
      bin = 5
    elif bigram_count < 128:
      bin = 6
    elif bigram_count < 256:
      bin = 7
    elif bigram_count < 512:
      bin = 8
    elif bigram_count < 1500:
      bin = 9
    elif bigram_count < 3500:
      bin = 10
    elif bigram_count < 7150:
      bin = 11
    elif bigram_count >= 7150:
      bin = 12
    else:
      print("error",bigram_count)
    vector_uni[bin].append(uni)
    vector_bia[bin].append(bia)
    vector_tri[bin].append(tria)

    if uni >1 or bia > 1 or tria > 1:
      print("error:", vector_uni[index], vector_bia[index], vector_tri[index], bigrams.get((word_2, word_3)), bigrams.get(word_2))
  print([[len(vector_uni[i]), len(vector_bia[i]), len(vector_tri[i])] for i in range(bins)])
  return vector_uni, vector_bia, vector_tri

def minimize_c_bins(vector_uni_big, vector_bia_big, vector_tri_big, voc_size=10000):
  bins = 13
  best_cs = np.zeros((bins, 2))
  for bin in range(bins):
    per, cs = minimize_perplexity(np.array(vector_uni_big[bin]), np.array(vector_bia_big[bin]), np.array(vector_tri_big[bin]))
    print(cs, per)
    best_cs[bin] = cs
  return best_cs

def get_perplexity_bins(best_cs, vector_uni_big, vector_bia_big, vector_tri_big, bins = 5):
  summa = 0
  length = 0
  for bin in range(bins):
    summa += np.sum(np.log(best_cs[bin, 0] * np.array(vector_tri_big[bin]) + best_cs[bin, 1] * np.array(vector_bia_big[bin]) + (1 - best_cs[bin, 0] - best_cs[bin, 1]) * np.array(vector_uni_big[bin])))
    length += len(vector_uni_big[bin])
  #print(summa, length)
  return np.exp(-summa/length)

In [ ]:
vector_uni_binned, vector_bia_binned, vector_tri_binned = get_ngrams_prob_vector_with_map(valid_set_int, unigrams_v, bigrams_v, trigrams_v)

best_cs_binned = minimize_c_bins(vector_uni_binned, vector_bia_binned, vector_tri_binned)


[[15113, 15113, 15113], [9111, 9111, 9111], [4569, 4569, 4569], [11003, 11003, 11003], [6153, 6153, 6153], [5154, 5154, 5154], [4533, 4533, 4533], [4246, 4246, 4246], [3749, 3749, 3749], [4181, 4181, 4181], [2417, 2417, 2417], [2334, 2334, 2334], [1195, 1195, 1195]]
[np.float64(0.04249999999999987), np.float64(0.545)] 253.36597958836043
[np.float64(0.15500000000000003), np.float64(0.53)] 195.1593349025002
[np.float64(0.21250000000000002), np.float64(0.5125)] 175.57368641688603
[np.float64(0.2825), np.float64(0.4625)] 160.8981494411517
[np.float64(0.36749999999999994), np.float64(0.395)] 140.37239553345924
[np.float64(0.4024999999999999), np.float64(0.3925)] 137.85809025006276
[np.float64(0.4375), np.float64(0.355)] 165.41806582075128
[np.float64(0.495), np.float64(0.3325)] 138.17118767106479
[np.float64(0.495), np.float64(0.355)] 129.03531793064874
[np.float64(0.52), np.float64(0.3525)] 178.7138382384483
[np.float64(0.48), np.float64(0.36)] 280.95911903083123
[np.float64(0.485), np.flo

We se that for low count grams bigram have higher weight than trigram but for common words the more weight to trigram. Wights look good on a curve? Above the code make its principle as same. Instead we use an obtimization on a lista instead to make c. As you see each function is build here on onther and therefore I think the code is selfexplained in combination with your instructions. I tried many bins and lookt att the distribution of groups and to make the cs look good an linear no har limits. Worth to note that many words with count 6000 count more on bigram than trigrams.

In [ ]:


vector_uni_t_binned, vector_bia_t_binned, vector_tri_t_binned = get_ngrams_prob_vector_with_map(test_set_int, unigrams_v, bigrams_v, trigrams_v)

best_perplexity = get_perplexity_from_probs(get_prob_tri(test_set_int, best_cs_binned, unigrams_v, bigrams_v, trigrams_v))

print(f"cs {best_cs_binned}, perplexity test: {best_perplexity}")

[[15594, 15594, 15594], [10128, 10128, 10128], [5416, 5416, 5416], [12656, 12656, 12656], [7109, 7109, 7109], [5864, 5864, 5864], [5181, 5181, 5181], [4800, 4800, 4800], [4310, 4310, 4310], [4664, 4664, 4664], [2933, 2933, 2933], [2654, 2654, 2654], [1119, 1119, 1119]]
cs [[0.0425 0.545 ]
 [0.155  0.53  ]
 [0.2125 0.5125]
 [0.2825 0.4625]
 [0.3675 0.395 ]
 [0.4025 0.3925]
 [0.4375 0.355 ]
 [0.495  0.3325]
 [0.495  0.355 ]
 [0.52   0.3525]
 [0.48   0.36  ]
 [0.485  0.39  ]
 [0.965  0.005 ]], perplexity test: 169.2874248994986


# 4

In [ ]:


def get_cache_tri(words, probs_tria, lambda1=0.05, m=100):
  latest = {}
  latest[words[0]] = latest.get(words[0], 0) + 1
  latest[words[1]] = latest.get(words[1], 0) + 1

  probabilities = np.zeros(len(words) -2, dtype=np.float32)

  for index in range(len(words) - 2):
    word_3 = words[index + 2]
    probabilities[index] = (probs_tria[index] * (1 - lambda1) + latest.get(word_3, 0)/min(m, index+2) * (lambda1))
    latest[word_3] = latest.get(word_3, 0) + 1
    if index >= m-2:
      latest[words[index + 2 - m]] -= 1
  return probabilities


I added above a lambda to give probabilities to word in last seequence a bit higher probability and below only run the code to se were it gave best result which. Below is a type of validationscore applied. Obove do we use a set of counts for measure the freq of the lastest words.

Below we otimize the the values of lambda and m gready (we have already found them)

In [ ]:
vector_prob_tri_vali = get_prob_tri(valid_set_int, best_cs_binned, unigrams_v, bigrams_v, trigrams_v)
print(f"perplexity tri valid: {get_perplexity_from_probs(vector_prob_tri_vali)}")
best_per = 250
for cache_length in np.arange(220, 280, 1): # smaller intervall wen we have already optimized around it for a greater set
  for lambda1 in np.arange(0.08, 0.13, 0.005):

    probabilities_cache_tri_valid =  get_cache_tri(valid_set_int, vector_prob_tri_vali, lambda1, cache_length)
    per = get_perplexity_from_probs(probabilities_cache_tri_valid)
    if per < best_per:
      best_per = per
      best_lambda = lambda1
      best_cache = cache_length
      print(f" best lambda = {best_lambda:.3f}, lengthword, {best_cache} \t perplexity: {best_per}")

print(f" best lambda = {best_lambda:.3f}, lengthword, {best_cache} \t perplexity: {best_per}")

  # best lambda = 0.105, m = 250

perplexity tri valid: 180.059016973229
 best lambda = 0.080, lengthword, 220 	 perplexity: 158.1709442138672
 best lambda = 0.085, lengthword, 220 	 perplexity: 158.06863403320312
 best lambda = 0.090, lengthword, 220 	 perplexity: 157.99667358398438
 best lambda = 0.095, lengthword, 220 	 perplexity: 157.95289611816406
 best lambda = 0.100, lengthword, 220 	 perplexity: 157.934814453125
 best lambda = 0.100, lengthword, 224 	 perplexity: 157.93165588378906
 best lambda = 0.100, lengthword, 226 	 perplexity: 157.92849731445312
 best lambda = 0.100, lengthword, 227 	 perplexity: 157.92315673828125
 best lambda = 0.100, lengthword, 228 	 perplexity: 157.91244506835938
 best lambda = 0.100, lengthword, 230 	 perplexity: 157.90380859375
 best lambda = 0.100, lengthword, 239 	 perplexity: 157.90025329589844
 best lambda = 0.105, lengthword, 239 	 perplexity: 157.89942932128906
 best lambda = 0.100, lengthword, 240 	 perplexity: 157.89137268066406
 best lambda = 0.105, lengthword, 240 	 perp

run test on the test and print from previus step

In [ ]:
vector_prob_tri_test = get_prob_tri(test_set_int, best_cs_binned, unigrams_v, bigrams_v, trigrams_v)
print(f"perplexity tri test (step 3): {get_perplexity_from_probs(vector_prob_tri_test):.3f}")

probabilities_cache_tri_test =  get_cache_tri(test_set_int, vector_prob_tri_test, 0.105, 257)
test_perplexity_cache_tri = get_perplexity_from_probs(probabilities_cache_tri_test)
print(f"test perplexity with cache (step 4): {test_perplexity_cache_tri:.3f}")


perplexity tri test (step 3): 169.287
test perplexity with cache (step 4): 151.445


I'm 16.44 away from what you said but I got an improvement with 20 for with chache.

With optimizing c binns on test_data I achived only 160 best test perplexity for trigrams and with good data. I tested for fun with test data only for see what is teoritical possible without validationdata for seeing if validationdata overfitted the underlaying and found that it didn't this didn't affected the model but it's why I contacted you so I don't need to make any more stuff that you havn't said.

Good improvement see we now with about 151.445 as perplexity of test.

# 5

Here we have code to predict the lstm next sequence and the ngram. We don't provide any information to lstm which is very very bad in preformence. Which make in much worse. We have also some function below that generates probablistic ngrams from a bigram of common words and we manual check so it's logical. It make it a bit better later and we add a temperatur and make nondeterministic chosses o lstm to see what will be generated, more randomly. The same for ngrams also with probabilities. Lstm will else return without context mostly dots or something like that if deterministic. The two function below is generators. We use probability of the 200 word with highest probability with temperature. If we use with deterministic we obtain shorter sentences for LSTM and more determinictic. I didn't know what do chosse.

In [ ]:
def predict_lstm(model, token):
  # very importtent # obs here we need next token
  model.eval()
  seq_len = np.shape(token)[0]

  token = batchify_full(torch.tensor(token, dtype=torch.long), seq_len)

  hidden = model.init_hidden(token.size(0), device)
  with torch.no_grad():
            data = token[:, :seq_len]
            data = data.to(device)

            if isinstance(hidden, tuple):
                hidden = (hidden[0].detach(), hidden[1].detach())
            else:
                hidden = [(h.detach(), c.detach()) for (h, c) in hidden]

            output, hidden, _ = model(data, hidden)
            return output.detach().cpu().numpy()
  return "error"

def generate_word_lstm(local_model, words, id_to_word, max_len_generate=30, deterministic=False):

    tokens = words.copy()

    for _ in range(max_len_generate):
        tokens_input = np.array(tokens)[-max_len_generate:]

        preds = predict_lstm(local_model, tokens_input)
        preds = preds[0]
        preds = np.exp(preds)
        probs = preds/np.sum(preds)
        if deterministic:
          next_id = np.argmax(probs)
        else:
          top_idx = np.argsort(probs)[-200:] # only include most 1000 common words
          filitered_probs = np.array([probs[i] for i in top_idx])**(0.75) #without this the probability will be very high for same words because
          next_id = np.random.choice(top_idx, p=filitered_probs/np.sum(filitered_probs))

        if id_to_word[next_id] == "<eos>":
          break

        tokens = np.array(tokens.tolist()+ [next_id])

    word_out = [id_to_word[t] for t in tokens]
    return " ".join(word_out) + "."

def generate_word_ngrams(phrase, id_to_word, c_values, lambda1 = 0.105, m = 250, max_len_generate=30, voc_size = 10000, deterministic=False):
    words = np.arange(0,voc_size-0.5, 1, dtype=np.int32)
    cache_array = {}
    tokens = phrase.copy()
    for word in tokens:
      cache_array[word] = cache_array.get(word, 0) + 1

    for index in range(max_len_generate):

        probs = np.zeros(voc_size)
        word1 = tokens[-2]
        word2 = tokens[-1]
        tri_prob_sum = 0
        for word3 in words:
          tri_prob = get_prob_tri([word1, word2, word3], c_values, unigrams_v, bigrams_v, trigrams_v, voc_size=voc_size)[0]
          tri_prob_sum += tri_prob
          probs[word3] = tri_prob * (1 - lambda1) + cache_array.get(word3, 0)/(max(3, min(tokens.shape[0], m))) * (lambda1)
        #print(np.sum(probs), tri_prob_sum)
        probs = probs/np.sum(probs)
        if deterministic:
          next_id = np.argmax(probs)
        else:
          top_idx = np.argsort(probs)[-200:] # only include most 1000 common words
          filitered_probs = np.array([probs[i] for i in top_idx])#**(0.5) no temeratuure
          next_id = np.random.choice(top_idx, p=filitered_probs/np.sum(filitered_probs))

        if id_to_word[next_id] == "<eos>":
           word_out = [id_to_word[t] for t in tokens]
           return " ".join(word_out)+"."
        cache_array[next_id] = cache_array.get(next_id, 0) + 1
        tokens = np.array(tokens.tolist()+ [next_id])
        #print("\t\t"+" ".join([id_to_word[t] for t in tokens])+" ...")
    word_out = [id_to_word[t] for t in tokens]
    return " ".join(word_out)

Function to get fivegrams mm and get most common fivegrams but we used instead fivegrams from biagram and tiagram

In [ ]:
def get_fivegrams(words):
    length_word = len(words)

    fivegrams = Counter()
    for i in range(length_word  - 4):
        possible = True
        for j in range(5):
          #remove numbers and dots for making most common five grams
          if words[i + j] == 45 or words[i + j] == 6142:
            possible = False
            break
        if possible:
          fivegrams[(words[i], words[i+1], words[i+2], words[i+3], words[i+4])] += 1

    return fivegrams

def get_most_common_fivegrams(fivegrams):
  return fivegrams.most_common(30)

def get_biagrams(words):
    length_word = len(words)

    biagrams = Counter()
    for i in range(length_word  - 1):
        possible = True
        for j in range(2):
          #remove dots for making most common one grams
          if  biagrams[i + j] == 43:
            possible = False
            break
        if possible:
          biagrams[(words[i], words[i+1])] += 1

    return biagrams

def get_trigrams(words):
    length_word = len(words)

    trigrams = Counter()
    for i in range(length_word  - 2):
        possible = True
        for j in range(3):
          #remove dots for making most common one grams
          if  trigrams[i + j] == 43:
            possible = False
            break
        if possible:
          trigrams[(words[i], words[i+1], words[i+2])] += 1

    return trigrams

def top_next_words(word, word2, trigrams, n_returns):
    # Collect all bigrams where word is the first word
    next_words = Counter()
    for (w1, w2, w3), freq in trigrams.items():

        if w1 == word and w2 == word2 and w1 != 43 and w2 != 43:
          if  w3 != 43:
            next_words[w3] += freq

    return next_words.most_common(n_returns)

def get_fivegrams_from_most_common_biagrams(biagrams, triagrams):
  fivegrams = np.zeros((30,5), dtype=np.int64)
  for i in range(30):
    while fivegrams[i, 4] == 0:
      fivegrams[i, :2] = biagrams.most_common(7000)[np.random.randint(7000)][0]
      for j in range(2, 5):
        next_word = top_next_words(fivegrams[i, j - 2], fivegrams[i, j - 1], triagrams, 10)
        if len(next_word) == 0:
          break
        fivegrams[i, j] = next_word[np.random.randint(len(next_word))][0]
  return fivegrams

In [ ]:
print(map_word_to_id["<eos>"])
ten_common_sentences = get_fivegrams_from_most_common_biagrams(get_biagrams(training_set_int), get_trigrams(training_set_int))
#ten_common_sentences = get_most_common_fivegrams(get_fivegrams(training_set_int))

43


give proposial sentences and below we will predict some of thos sentences.

In [ ]:
ten_common_sentences_np_array = np.array([np.array(one_sentence) for one_sentence in ten_common_sentences])
sentences_word = [" ".join([map_id_to_word[int(word)] for word in sentence]) for sentence in ten_common_sentences_np_array][:30]
sentences_int = ten_common_sentences_np_array[:30]
print("most common 5 words")
for i in range(len(sentences_word)):
  print(i, sentences_word[i])


most common 5 words
0 court in a letter to
1 have N plants and a
2 part because of expectations of
3 basis points above treasury notes
4 called for average americans to
5 will get <unk> <unk> to
6 in terms such as a
7 is considered likely now but
8 unsecured creditors and anticipate filing
9 will begin to accelerate because
10 december N galileo will <unk>
11 bank 's retail activities in
12 face of competition spurred by
13 late friday including higher margins
14 shareholders ' syndicate that supplies
15 of more choice outweigh the
16 federal funds interest rate to
17 and in july a few
18 weakness in steel marking what
19 the states go down industrial
20 they said stock-index arbitrage trades
21 including $ N an amount
22 invested in stocks they can
23 is based said the <unk>
24 N cars equipped with a
25 by major employers to pay
26 effort to get out he
27 would go up because of
28 <unk> to make sure his
29 raise the N <unk> and


In [ ]:
# choossing good sentences from above
indexs_to_generate_from = [7, 27, 24 , 4, 11, 16, 17, 20, 14, 29]

In [ ]:
for epoch, i in enumerate(indexs_to_generate_from):
  print(f'Generating sentence "'+ sentences_word[i] + '" ...')
  print("\tLSTM:")
  for j in range(10):
    lstm_sentence = generate_word_lstm(model, sentences_int[i], map_id_to_word)

    print(f"\t\t"+ lstm_sentence)
  # few epoch for ngrams thus very long time to predict (ordo(vocab_size)) longer time than train lstm
  if epoch<3:
    print("\tNgrams")
    for j in range(5):
      ngram_sentence = generate_word_ngrams(sentences_int[i], map_id_to_word, best_cs_binned)
      print(f"\t\t"+ ngram_sentence)


Generating sentence "is considered likely now but" ...
	LSTM:
		is considered likely now but after because most just N subject at on an building u.s. for $ enough likely a too no a in more on part this a far.
		is considered likely now but made an a up $ high that a one to to a many after just to getting for further an one less in more less if american.
		is considered likely now but another far N N in u.s. the <unk> running much to <unk> <unk> the either and even for on being even quite a used and to the we N through.
		is considered likely now but on more and so particularly <unk> a an most a <unk> expected <unk> <unk> by now under of some the a the under n't keeping that national <unk> known still.
		is considered likely now but a where the a and said.
		is considered likely now but a whether a a the a n't the a known of an N about a supposed <unk> are the far he a n't common n't more the the <unk> earlier.
		is considered likely now but providing far and <unk> moving n't a they a t

Lstm given most on context but ngrams given by the procedding words. Problem is that some words is quite common in and therefore gives probobalistic words. Importent note is that often does lstm generate same word as proceding even if its always false but its generated by context. Ngram good to generate probobalistic words but lstm can found the context easy. We set the temperature to (0.75) and limmit it to 200 most prob words for both ngrams and lstm. Ngram better next word but will lack context lstm have context. The word is often very common by the words behind but it will produce very bad longer sentences.

Clear stuff is that LSTM generates by context therefore some worser order but it generate words in a context where instead ngram generates words that should by looking at the past 10 words.

LSTM keep context always, while ngram give common words by the last few.

We could use without a temperature LSTM and get more clearer senteces but then we have a problem. Because of the reinforcement of < unk > will many words be unknow or common words that reapet many times or a very likly word. Because word is based on context and therefor will it happen. ngram takes long time to generate thus we need to check all probabilities from words in vocabular not so great.

Lstm generates quite good contextbased next words. Need more context then ngrams so lstm generates very good in end of sentence by context but worser in beginning of sentence than ngrams. Given 30 words before lstm:s is much better than ngrams. (We used length of 32 in lstm model) so therefore lstm is not so great by a context of 5 words in comperision, but with about more words the lstm predicts quite good.

Ngram is a more local model and Lstm:s a more context based model. The context's is very importent to achive low perplexity (73.5) but without much contet hard to generate good words. Ngrams makes very logical short segment but if we look at last few words compare to the first words it will lack context.


I generated 30 sentences (of 5 words) and chossen the sentences that could possible be a sentence with right context and more.


LSTM does'n divigerar in words but n-grams does. Below is sentences generated but in plain form.

Note that we have use non deterministic ngrams and LSTM need more context.

# Sequences generated

## Generating sentence "is considered likely now but" ...
#### LSTM:
    1. is considered likely now but after because most just N subject at on an building u.s. for $ enough likely a too no a in more on part this a far.
    
    2. is considered likely now but made an a up $ high that a one to to a many after just to getting for further an one less in more less if american.
    3. is considered likely now but another far N N in u.s. the < unk > running much to < unk > < unk > the either and even for on being even quite a used and to the we N through.
    4. is considered likely now but on more and so particularly < unk > a an most a < un k> expected < unk > < unk > by now under of some the a the under n't keeping that national < unk > known still.
    5. is considered likely now but a where the a and said.
    6. is considered likely now but a whether a a the a n't the a known of an N about a supposed < unk > are the far he a n't common n't more the the < unk > earlier.
    7. is considered likely now but providing far and < unk > moving n't a they a the necessary the last N mr. < unk > there that from < unk > up < unk > limited being seeking in a during and would.
    8. is considered likely now but even the an which a < unk > < unk > a < unk > how part one an the under much common even most more an a mr. both close the the there are while.
    9. is considered likely now but an nearly.
    10. is considered likely now but three to < unk > starting moving rising working be the an < unk > when to a his right not a over part what its a considered like < unk > < unk > to the in.


#### Ngrams

    1. is considered likely now but is a is last is a is.
    2. is considered likely now but two company.
    3. is considered likely now but the largest penalty ever since the drought could drop by N european media play and other benefits.
    4. is considered likely now but it is increasing capital mr. stoltzman chooses to is scheduled to to of n't of much capacity of in N.
    5. is considered likely now but would be limited < unk > < unk > according limited is but < unk > senior trading at < unk > inc.


## Generating sentence "would go up because of" ...

  
#### LSTM:
		1. would go up because of buy < unk > handle be develop retain work at hold to challenge contribute find result receive do in come it develop determine be carry support be keep directly and that those.
		2. would go up because of report increase do lead < unk > < unk > be go the be on < unk > are contribute be eventually maintain know stop be try reduce start explain < unk > carry out foreign he federal.
		3. would go up because of seem like to all pay go an < unk > accept make be help change mr. < unk > allow like a < unk > be have accept make determine < unk > prove part.
		4. would go up because of add make have < unk > be.
		5. would go up because of be further hold look represent find < unk > be probably depend be help be generate go accept pass know want no help all find it keep join to we their some.
		6. would go up because of have bring < unk > have replace be n't not keep come buy be < unk > be be < unk > permit be act do do reduce the find be an N across japan < unk >.
		7. would go up because of afford be ease that produce tell replace be all have act have seem have trade also be include have they < unk > < unk > n't soon have eliminate less three to first.
		8. would go up because of eliminate continue for of cause hold n't of show have < unk > have follow grow be < unk > go be < unk > cost come apply < unk > be move not when a there a.
		9. would go up because of seek < unk > give support work have one boost show become look affect < unk > report n't add be become stop < unk > not < unk > grow keep return provide away around < unk > san.
		10. would go up because of go be for < unk > more < unk > not look in have slow see happen run match hurt protect < unk > pay be have explain be be take want behind that of the.

#### Ngrams
		1. would go up because of the $ of several.
		2. would go up because of < unk > of is years from trading is < unk >.
		3. would go up because of its latest fiscal N cents on a a this < unk > into a recession is more than half the rate.
		4. would go up because of its N million dividend dividend for a few individual retirement of that has gained N p.m. est the the year dividend payable dec. N N of the the new go
		5. would go up because of a major areas down a big iron mill down go to complete details and the a a strategic planners iron & < unk > in such things to strategic equation N of

##Generating sentence "N cars equipped with a" ...
#### LSTM:
		1. N cars equipped with a < unk > the < unk > N federal its big workers of restaurants < unk > units billion by < unk > big sales < unk > < unk > americans < unk > lire is former billion major had but < unk > takeover.
		2. N cars equipped with a market < unk > will of be soviet mr. barrels < unk >.
		3. N cars equipped with a corporate months executive other tax to california < unk > pages months < unk > local of N < unk > senior < unk > sales are N to N but of yen < unk > that between mr. year.
		4. N cars equipped with a of years < unk > million.
		5. N cars equipped with a to japanese or third the largest.
		6. N cars equipped with a p.m. to < unk > N common ounces square people the < unk > < unk > square private stocks of < unk > < unk > the N has < unk > acres mr. the big office not.
		7. N cars equipped with a acres < unk > states other the companies.
		8. N cars equipped with a election miles million for have to million < unk > key quarter has large recent in years board major but will children people < unk > world < unk > < unk > N the.
		9. N cars equipped with a days < unk > first of black of years million 's < unk > the < unk > day of N series members p.m. customers to government of corporate < unk > years more ' < unk > control company.
		10. N cars equipped with a had N local N level < unk > million yen for < unk > but N monthly million were to san N at an < unk > < unk > years N N and such his an former.
## Ngrams
		1. N cars equipped with a more cars he says which to yield dropped N to N units to < unk > a certain level and a slight loss as and others get much as < unk > more heavily
		2. N cars equipped with a computer services company of repeatedly repeatedly that rubbermaid inc. a < unk > of trading to be the one that with all the < unk > of the of the talks.
		3. N cars equipped with a federal judge < unk > federal at judge in manhattan cable announced that mitsubishi down N < unk > or face.
		4. N cars equipped with a N N indicates that were produced the movement N to $ N million according to < unk > corp. called N indicates and the fact.
		5. N cars equipped with a N with a year-ago quarter.